# schedule backtest contract notify v1

目标：验证 dashboard refresh 的详细通知会带上 backtest contract schema、walk-forward fold 数和滑点配置，方便 review 时发现配置漂移。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

EXPERIMENT = "schedule_backtest_contract_notify_v1"
CARD = ROOT / "research" / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)

In [2]:
events: list[str] = []

def matrix_runner(**kwargs):
    card_dir = Path(kwargs["card_dir"])
    card_dir.mkdir(parents=True, exist_ok=True)
    return {"variant_count": 2, "status": "feature_set_matrix_completed"}


def dashboard_exporter(card_dir: Path, *, output_path: Path, top_n: int):
    payload = {
        "schema_version": "dashboard.model_matrix.v1",
        "artifact_id": EXPERIMENT,
        "top_models": [{"model": "ridge_mse", "rank": 1}],
        "manifest_source": {
            "kind": "analysis_manifest",
            "path": str(card_dir / "analysis_manifest.json"),
            "schema_version": "artifact.v1",
            "manifest_hash": "d" * 64,
            "producer": "src.analysis.exports",
        },
        "backtest": {
            "config": {
                "schema_version": "backtest.config.v1",
                "start": "2026-05-01",
                "end": "2026-06-08",
                "cost_model": {
                    "open_cost_bps": 1.0,
                    "close_cost_bps": 1.0,
                    "slippage_bps": 1.5,
                },
            },
            "walk_forward_splits": [
                {"fold": 1, "train_start": "2026-01-01", "train_end": "2026-04-30", "test_start": "2026-05-01", "test_end": "2026-05-13"},
                {"fold": 2, "train_start": "2026-01-14", "train_end": "2026-05-13", "test_start": "2026-05-14", "test_end": "2026-05-26"},
            ],
        },
    }
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return payload


config = ModelDashboardPipelineConfig(
    name="model_dashboard_refresh",
    card_dir=CARD / "matrix_card",
    dashboard_output_path=CARD / "dashboard_export.json",
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=("ridge_mse",),
    notify_details=True,
)

result = run_model_dashboard_refresh_once(
    config,
    lock_dir=CARD / "locks",
    notify=events.append,
    matrix_runner=matrix_runner,
    dashboard_exporter=dashboard_exporter,
)

summary = {
    "schema_version": "research.schedule_backtest_contract_notify_v1",
    "status": result.status,
    "payload": result.payload,
    "notification": events[0] if events else "",
    "notification_contains_backtest_contract": "backtest=backtest.config.v1 folds=2 slippage_bps=1.5" in (events[0] if events else ""),
}
(CARD / "notification_events.json").write_text(json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8")
(CARD / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary

{'schema_version': 'research.schedule_backtest_contract_notify_v1',
 'status': 'success',
 'payload': {'matrix_card': '/Users/rziiiii/dev/workspace/research/schedule_backtest_contract_notify_v1/matrix_card',
  'dashboard_export': '/Users/rziiiii/dev/workspace/research/schedule_backtest_contract_notify_v1/dashboard_export.json',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'dashboard_artifact_id': 'schedule_backtest_contract_notify_v1',
  'dashboard_manifest_kind': 'analysis_manifest',
  'dashboard_manifest_path': '/Users/rziiiii/dev/workspace/research/schedule_backtest_contract_notify_v1/matrix_card/analysis_manifest.json',
  'dashboard_manifest_hash': 'dddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddddd',
  'dashboard_manifest_producer': 'src.analysis.exports',
  'dashboard_page': '',
  'dashboard_page_schema_version': '',
  'dashboard_view_model_schema_version': '',
  'dashboard_page_model_card_count': 0,
  'top_model': 'ridge_mse',
  'attribution_card': 

In [3]:
assert result.status == "success"
assert summary["payload"]["backtest_schema_version"] == "backtest.config.v1"
assert summary["payload"]["walk_forward_fold_count"] == 2
assert summary["payload"]["backtest_slippage_bps"] == 1.5
assert summary["notification_contains_backtest_contract"] is True
assert (CARD / "dashboard_export.json").exists()
assert (CARD / "notification_events.json").exists()

## 中文结论

- 做了什么：验证 `src/schedule` 的 dashboard refresh 详细通知会读取 dashboard export 里的 `backtest.config` 和 `walk_forward_splits`。
- 核心指标：通知中包含 `backtest=backtest.config.v1 folds=2 slippage_bps=1.5`，payload 同时保留 `backtest_start`、`backtest_end` 和成本字段。
- regime 差异：本 notebook 不做模型 regime 对比；它验证 schedule 通知可以携带支撑 regime/model review 的 backtest 配置上下文。
- 是否晋升：晋升为 schedule 通知框架能力；不代表任何策略晋升 production。
- 风险：通知只展示摘要，完整成本、仓位和 walk-forward 明细仍以 dashboard manifest/export 为准。
- 下一步：把 objective/promotion score 也纳入 manifest 和 dashboard refresh 摘要。